In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Input dan output Estimator

<Accordion>
<AccordionItem title="Versi paket">

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Halaman ini memberikan gambaran umum input dan output primitif Qiskit Runtime Estimator, yang mengeksekusi beban kerja pada sumber daya komputasi IBM Quantum&reg;. Estimator memungkinkan kamu mendefinisikan beban kerja yang divektorisasi secara efisien menggunakan struktur data yang disebut [**Primitive Unified Bloc (PUB)**](/guides/primitive-input-output#pubs). PUB digunakan sebagai input ke metode [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) untuk primitif Estimator, yang mengeksekusi beban kerja yang didefinisikan sebagai sebuah job. Kemudian, setelah job selesai, hasilnya dikembalikan dalam format yang bergantung pada PUB yang digunakan serta opsi runtime yang ditentukan dari primitif.

## Input
Setiap PUB memiliki format ini:

(`<circuit tunggal>`, `<satu atau lebih observable>`, `<opsional satu atau lebih nilai parameter>`, `<opsional presisi>`),

`Nilai parameter` opsional bisa berupa daftar atau parameter tunggal. Elemen dari observable dan nilai parameter digabungkan dengan mengikuti aturan broadcasting NumPy seperti yang dijelaskan dalam topik [Input dan output primitif](primitive-input-output#broadcasting-rules), dan satu estimasi nilai ekspektasi dikembalikan untuk setiap elemen dari bentuk yang di-broadcast.

> **Note:** Jika input mengandung pengukuran, pengukuran tersebut diabaikan.

Untuk primitif Estimator, sebuah PUB dapat mengandung paling banyak empat nilai:
- Satu `QuantumCircuit` tunggal, yang mungkin mengandung satu atau lebih objek [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Daftar satu atau lebih observable, yang menentukan nilai ekspektasi yang akan diestimasi, disusun dalam sebuah array (misalnya, satu observable direpresentasikan sebagai array 0-d, daftar observable sebagai array 1-d, dan seterusnya). Datanya bisa dalam salah satu format `ObservablesArrayLike` seperti `Pauli`, `SparsePauliOp`, `PauliList`, atau `str`.
   > **Note:** - Observable yang komutasi **dalam PUB yang sama** dikelompokkan bersama menggunakan [metode ini](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting).
>    - Observable yang komutasi dalam PUB yang berbeda, meskipun memiliki circuit yang sama, tidak diestimasi menggunakan pengukuran yang sama. Setiap PUB merepresentasikan basis yang berbeda untuk pengukuran, dan oleh karena itu, pengukuran terpisah diperlukan untuk setiap PUB.
>    - Untuk memastikan bahwa observable yang komutasi diestimasi menggunakan pengukuran yang sama, kelompokkan mereka dalam PUB yang sama.
- Kumpulan nilai parameter untuk diikat ke circuit. Ini dapat ditentukan sebagai satu objek array-like di mana indeks terakhir adalah untuk objek `Parameter` circuit atau dihilangkan (atau setara, diatur ke `None`) jika circuit tidak memiliki objek `Parameter`.
- (Opsional) Presisi target untuk nilai ekspektasi yang akan diestimasi
---

Kode berikut menunjukkan contoh set input yang divektorisasi ke primitif `Estimator` dan mengeksekusinya pada backend IBM&reg; sebagai satu objek `RuntimeJobV2`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## Output
Setelah satu atau lebih PUB dikirim ke QPU untuk eksekusi dan job berhasil diselesaikan, data dikembalikan sebagai objek kontainer [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) yang diakses dengan memanggil metode `RuntimeJobV2.result()`.

`PrimitiveResult` berisi daftar iterable dari objek [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) yang berisi hasil eksekusi untuk setiap PUB.

Setiap elemen dari daftar ini sesuai dengan setiap PUB yang dikirimkan ke metode `run()` primitif (misalnya, job yang dikirim dengan 20 PUB akan mengembalikan objek `PrimitiveResult` yang berisi daftar 20 objek `PubResult`, satu sesuai dengan setiap PUB).

Setiap `PubResult` untuk primitif Estimator setidaknya mengandung array nilai ekspektasi (`PubResult.data.evs`) dan deviasi standar terkait (baik `PubResult.data.stds` atau `PubResult.data.ensemble_standard_error` tergantung pada `resilience_level` yang digunakan), tetapi dapat mengandung lebih banyak data tergantung pada opsi mitigasi error yang ditentukan.

Setiap objek `PubResult` memiliki atribut `data` dan `metadata`.
    - Atribut `data` adalah [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) yang dikustomisasi dan berisi nilai pengukuran aktual, deviasi standar, dan sebagainya.
    - `DataBin` memiliki berbagai atribut tergantung pada bentuk atau struktur PUB terkait serta opsi mitigasi error yang ditentukan oleh primitif yang digunakan untuk mengirimkan job (misalnya, [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) atau [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - Atribut `metadata` berisi informasi tentang runtime dan opsi mitigasi error yang digunakan (dijelaskan kemudian di bagian [Metadata hasil](#result-metadata) di halaman ini).

Berikut adalah garis besar visual dari struktur data `PrimitiveResult` untuk output Estimator:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

Sederhananya, satu job mengembalikan objek `PrimitiveResult` dan berisi daftar satu atau lebih objek `PubResult`. Objek `PubResult` ini kemudian menyimpan data pengukuran untuk setiap PUB yang dikirimkan ke job.

Cuplikan kode di bawah ini menjelaskan format `PrimitiveResult` (dan `PubResult` terkait) untuk job yang dibuat di atas.

In [ ]:
print(
    f"The result of the submitted job had {len(result)} "
    f"PUBs and has a value:\n {result}\n"
)
print(
    "The associated PubResult of this job has the following data bins:\n "
    "{result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets"
    "having shape (100, 2), where 2 is the number of parameters in the "
    "circuit, combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        "The expectation values measured from this PUB are: \n"
        "{result[0].data.evs}\n"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
